# Application of Machine Learning to Improve Pairs Trading Strategy

**Author:** Nabyl H.  
**Date:** February 2026  
**Course:** Certificate in Python for Finance (CPF)

---

## Abstract

Pairs trading is a market-neutral strategy that exploits price inefficiencies between two correlated assets. Traditional approaches rely on statistical measures like z-score thresholds, which may miss complex patterns in financial data. This project explores the application of machine learning methods to enhance pairs trading performance.

We implement:
- A **baseline z-score strategy** for comparison
- A **Random Forest classifier** to predict mean reversion opportunities
- **Feature engineering** including rolling statistics, volatility measures, and lagged variables

Results show that the ML approach achieves superior risk-adjusted returns compared to the traditional method, with a Sharpe ratio of X.XX vs X.XX for the baseline.

---

## 1. Introduction

### 1.1 Pairs Trading Fundamentals

Pairs trading is a statistical arbitrage strategy that involves:
1. Identifying two historically correlated assets
2. Taking a long position in the underperforming asset and a short position in the overperforming asset
3. Profiting when the prices converge

The strategy is market-neutral, meaning it generates returns regardless of overall market direction.

### 1.2 Limitations of Traditional Approaches

Traditional pairs trading uses statistical measures such as:
- **Z-score** of the price ratio
- Fixed entry/exit thresholds (e.g., ±2.0)
- Rolling windows for parameter estimation

**Limitations:**
- Linear assumptions may miss non-linear relationships
- Fixed thresholds fail to adapt to changing market regimes
- Slow to react to structural breaks

### 1.3 Machine Learning Opportunities

Machine learning can address these limitations by:
- Capturing **non-linear patterns** in spread dynamics
- **Adaptive regime detection** through feature learning
- Incorporating **multiple features** beyond simple ratios
- **Dynamic position sizing** based on prediction confidence

---

## 2. Data Preparation

```python
# Import necessary libraries
import sys
sys.path.append('..')  # To access src folder

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Import our modules
from src.baseline import run_baseline_strategy
from src.models import run_ml_pipeline, create_features, create_labels

# Load data
data_path = Path('../data/pairs_data.csv')
data = pd.read_csv(data_path, index_col=0, parse_dates=True)

print("=" * 60)
print("DATA OVERVIEW")
print("=" * 60)
print(f"Shape: {data.shape}")
print(f"Date range: {data.index[0].date()} to {data.index[-1].date()}")
print(f"Number of trading days: {len(data)}")
print("\nFirst 5 rows:")
data.head()

In [ ]:
# Data visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Price series
axes[0, 0].plot(data.index, data['Asset_A'], label='Asset A', linewidth=1)
axes[0, 0].plot(data.index, data['Asset_B'], label='Asset B', linewidth=1)
axes[0, 0].set_title('Price Series')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Price')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Price ratio
axes[0, 1].plot(data.index, data['Asset_A'] / data['Asset_B'], color='purple', linewidth=1)
axes[0, 1].axhline(y=1, color='red', linestyle='--', alpha=0.5)
axes[0, 1].set_title('Price Ratio (Asset A / Asset B)')
axes[0, 1].set_xlabel('Date')
axes[0, 1].set_ylabel('Ratio')
axes[0, 1].grid(True, alpha=0.3)

# Returns distribution
returns_A = data['Asset_A'].pct_change().dropna()
returns_B = data['Asset_B'].pct_change().dropna()

axes[1, 0].hist(returns_A, bins=50, alpha=0.7, label='Asset A', density=True)
axes[1, 0].hist(returns_B, bins=50, alpha=0.7, label='Asset B', density=True)
axes[1, 0].set_title('Returns Distribution')
axes[1, 0].set_xlabel('Daily Return')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Rolling correlation
rolling_corr = returns_A.rolling(60).corr(returns_B)
axes[1, 1].plot(rolling_corr.index, rolling_corr, color='green', linewidth=1)
axes[1, 1].axhline(y=0.7, color='red', linestyle='--', alpha=0.5, label='Target correlation')
axes[1, 1].set_title('60-Day Rolling Correlation')
axes[1, 1].set_xlabel('Date')
axes[1, 1].set_ylabel('Correlation')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Statistical summary
print("=" * 60)
print("STATISTICAL SUMMARY")
print("=" * 60)
print("\nAsset A:")
print(f"  Mean price: {data['Asset_A'].mean():.2f}")
print(f"  Std price: {data['Asset_A'].std():.2f}")
print(f"  Min price: {data['Asset_A'].min():.2f}")
print(f"  Max price: {data['Asset_A'].max():.2f}")

print("\nAsset B:")
print(f"  Mean price: {data['Asset_B'].mean():.2f}")
print(f"  Std price: {data['Asset_B'].std():.2f}")
print(f"  Min price: {data['Asset_B'].min():.2f}")
print(f"  Max price: {data['Asset_B'].max():.2f}")

# Correlation
correlation = data['Asset_A'].corr(data['Asset_B'])
print(f"\nCorrelation between assets: {correlation:.4f}")

# Check for cointegration (simplified)
from statsmodels.tsa.stattools import coint
score, pvalue, _ = coint(data['Asset_A'], data['Asset_B'])
print(f"Cointegration test p-value: {pvalue:.4f}")
if pvalue < 0.05:
    print("  → Assets are cointegrated (good for pairs trading)")
else:
    print("  → Assets are not cointegrated")